In [0]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("assignment1").getOrCreate()

DATA = "/Volumes/jaswanth/default/day-3_assignment/datasets"
OUT = "/tmp/aj/assignment1"   # on Databricks e.g. "/tmp/<yourname>/assignment1"

ORDERS_SCHEMA = """order_id INT, order_date DATE, customer_id INT,
                   order_status STRING, amount DOUBLE"""

orders = (spark.read.option("header", True)
          .schema(ORDERS_SCHEMA)
          .csv(f"{DATA}/orders.csv"))
orders.createOrReplaceTempView("orders")


In [0]:
orders.display()
orders.printSchema()

order_id,order_date,customer_id,order_status,amount
1,2026-08-16,26,COMPLETE,9355.49
2,2026-08-25,16,ON_HOLD,6633.45
3,2026-07-06,1,CLOSED,6646.6
4,2026-07-25,11,PENDING,2064.02
5,2026-08-16,45,COMPLETE,22397.18
6,2026-08-02,31,COMPLETE,13379.83
7,2026-07-26,16,COMPLETE,3398.49
8,2026-08-12,27,ON_HOLD,12634.97
9,2026-08-29,21,PROCESSING,9071.12
10,2026-08-04,5,COMPLETE,11528.82


root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- amount: double (nullable = true)



In [0]:
orders_infer = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{DATA}/orders.csv")

orders_infer.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- amount: double (nullable = true)



In [0]:
# Compare explicit vs inferred schema
print("EXPLICIT SCHEMA:")
orders.printSchema()

print("INFERRED SCHEMA:")
orders_infer.printSchema()

for f1, f2 in zip(orders.schema.fields, orders_infer.schema.fields):
    if f1.dataType != f2.dataType:
        print(f"DIFFERS: {f1.name} -> explicit={f1.dataType}, inferred={f2.dataType}")
    else:
        print(f"MATCH:   {f1.name} -> {f1.dataType}")

print("""
Spark UI jobs:
- Explicit schema: 1 job (reads data directly with known schema)
- inferSchema=True: 2 jobs (1 extra scan to detect types, then real read)
""")

EXPLICIT SCHEMA:
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- amount: double (nullable = true)

INFERRED SCHEMA:
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- amount: double (nullable = true)

MATCH:   order_id -> IntegerType()
MATCH:   order_date -> DateType()
MATCH:   customer_id -> IntegerType()
MATCH:   order_status -> StringType()
MATCH:   amount -> DoubleType()

Spark UI jobs:
- Explicit schema: 1 job (reads data directly with known schema)
- inferSchema=True: 2 jobs (1 extra scan to detect types, then real read)



In [0]:
dirty = f"{DATA}/orders_dirty.csv"

# PERMISSIVE
permissive = spark.read.option("header", True).schema(ORDERS_SCHEMA).option("mode", "PERMISSIVE").csv(dirty)
print("PERMISSIVE ->", len(permissive.collect()), "rows")
permissive.filter("order_date IS NULL OR amount IS NULL").show()

# DROPMALFORMED
dropped = spark.read.option("header", True).schema(ORDERS_SCHEMA).option("mode", "DROPMALFORMED").csv(dirty)
print("DROPMALFORMED ->", len(dropped.collect()), "rows")

# FAILFAST
failfast = spark.read.option("header", True).schema(ORDERS_SCHEMA).option("mode", "FAILFAST").csv(dirty)
print("FAILFAST ->", len(failfast.collect()), "rows")

PERMISSIVE -> 45 rows
+--------+----------+-----------+------------+-------+
|order_id|order_date|customer_id|order_status| amount|
+--------+----------+-----------+------------+-------+
|       4|      NULL|         11|     PENDING|2064.02|
|       8|2026-08-12|         27|     ON_HOLD|   NULL|
|      27|2026-08-02|         19|    COMPLETE|   NULL|
+--------+----------+-----------+------------+-------+

DROPMALFORMED -> 43 rows


---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-8743383949096942>, line 14
     12 # FAILFAST
     13 failfast = spark.read.option("header", True).schema(ORDERS_SCHEMA).option("mode", "FAILFAST").csv(dirty)
---> 14 print("FAILFAST ->", len(failfast.collect()), "rows")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2084, in DataFrame.collect(self)
   2083 def collect(self) -> List[Row]:
-> 2084     table, schema = self._to_table()
   2086     # not all datatypes are supported in arrow based collect
   2087     # here always verify the schema by from_arrow_schema
   2088     schema2 = from_arrow_schema(table.schema, prefer_timestamp_ntz=True)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2168, in DataFrame._to_table(self)
   2166 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:

In [0]:
# Task 3: Capture the bad rows with _corrupt_record
dirty = f"{DATA}/orders_dirty.csv"

bad = (spark.read.option("header", True)
       .schema(ORDERS_SCHEMA + ", _corrupt_record STRING")
       .option("mode", "PERMISSIVE")
       .option("columnNameOfCorruptRecord", "_corrupt_record")
       .csv(dirty))

rows = [r for r in bad.collect() if r["_corrupt_record"] is not None]
print("bad rows:", len(rows))
for r in rows:
    print(r["_corrupt_record"])

bad rows: 2
4,07/15/2026,11,PENDING,2064.02
8,2026-08-12,27,ON_HOLD,abc


In [0]:
# Task 4: Nested JSON
employees = spark.read.json(f"{DATA}/employees.json")

# schema is inferred
employees.printSchema()

employees.select("name",
                 F.col("address.city"),
                 F.col("address.state"),
                 F.size("skills").alias("n_skills")).show(truncate=False)

# employees per state
emp_by_state = employees.groupBy(F.col("address.state").alias("state")).count()
emp_by_state.orderBy(F.desc("count")).show()

# write as JSON 
OUT = "/Volumes/jaswanth/default/day-3_assignment/output"
emp_by_state.write.mode("overwrite").json(f"{OUT}/emp_by_state")
print("written to", f"{OUT}/emp_by_state")

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- pincode: string (nullable = true)
 |    |-- state: string (nullable = true)
 |-- certifications: struct (nullable = true)
 |    |-- aws: long (nullable = true)
 |    |-- azure: long (nullable = true)
 |    |-- databricks: long (nullable = true)
 |    |-- gcp: long (nullable = true)
 |-- dept: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- join_date: string (nullable = true)
 |-- manager_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)

+-------------+----------+-----+--------+
|name         |city      |state|n_skills|
+-------------+----------+-----+--------+
|Kavya Kumar  |Kolkata   |WB   |2       |
|Karthik Gupta|Pune      |MH   |4       |
|Ananya Iyer  |Chennai   |TN   |4       |
|Rohan Iyer   |Bengaluru |KA   |3       |
|Aditya Sharma|Ko

In [0]:
# Task 5: Parquet and save modes
OUT = "/Volumes/jaswanth/default/day-3_assignment/output"
orders_parquet = f"{OUT}/orders_parquet"

# first write
orders.write.mode("overwrite").parquet(orders_parquet)
print("first write --", spark.read.parquet(orders_parquet).count(), "rows")

# default mode is errorifexists
try:
    orders.write.parquet(orders_parquet)
except Exception:
    print("default mode -- ERROR: path already exists")
print("after default --", spark.read.parquet(orders_parquet).count(), "rows")

# append 
orders.write.mode("append").parquet(orders_parquet)
print("after append --", spark.read.parquet(orders_parquet).count(), "rows")

# ignore 
orders.write.mode("ignore").parquet(orders_parquet)
print("after ignore --", spark.read.parquet(orders_parquet).count(), "rows")

# overwrite
orders.write.mode("overwrite").parquet(orders_parquet)
print("after overwrite --", spark.read.parquet(orders_parquet).count(), "rows")

first write -- 300 rows
default mode -- ERROR: path already exists
after default -- 300 rows
after append -- 600 rows
after ignore -- 600 rows
after overwrite -- 300 rows


In [0]:
# Task 6: partitionBy on write
OUT = "/Volumes/jaswanth/default/day-3_assignment/output"
orders_by_status = f"{OUT}/orders_by_status"

orders.write.mode("overwrite").partitionBy("order_status").parquet(orders_by_status)

print("sub-folders:")
for f in dbutils.fs.ls(orders_by_status):
    print("  ", f.name)

closed = spark.read.parquet(orders_by_status).filter("order_status = 'CLOSED'")
print("CLOSED orders:", closed.count())
closed.explain()   # look for PartitionFilters -> Spark reads only the CLOSED folder

sub-folders:
   _SUCCESS
   order_status=CANCELED/
   order_status=CLOSED/
   order_status=COMPLETE/
   order_status=ON_HOLD/
   order_status=PENDING/
   order_status=PENDING_PAYMENT/
   order_status=PROCESSING/
   order_status=SUSPECTED_FRAUD/
CLOSED orders: 51
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonScan parquet [order_id#10371,order_date#10372,customer_id#10373,amount#10374,order_status#10375] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/jaswanth/default/day-3_assignment/output/orders_by_status], OptionalDataFilters: [], PartitionFilters: [isnotnull(order_status#10375), (order_status#10375 = CLOSED)], ReadSchema: struct<order_id:int,order_date:date,customer_id:int,amount:double>, RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
# Task 7: CSV with options -> pipe-separated single file
OUT = "/Volumes/jaswanth/default/day-3_assignment/output"
customers_pipe = f"{OUT}/customers_pipe"

# read source
customers = (spark.read
             .option("header", True)
             .option("inferSchema", True)
             .csv(f"{DATA}/customers.csv"))

print("source:", customers.count(), "rows")
customers.printSchema()

# write as ONE pipe-separated file with header
(customers.coalesce(1)
 .write.mode("overwrite")
 .option("sep", "|")
 .option("header", True)
 .csv(customers_pipe))

# confirm single output file
files = dbutils.fs.ls(customers_pipe)
csv_files = [f.name for f in files if f.name.endswith(".csv")]
print("output files:", csv_files)

# read it back with the same separator
back = (spark.read
        .option("header", True)
        .option("sep", "|")
        .option("inferSchema", True)
        .csv(customers_pipe))

print("read back:", back.count(), "rows")
print("columns:", back.columns)
assert back.count() == customers.count(), "row count mismatch!"
assert back.columns == customers.columns, "column mismatch!"
back.show(5, truncate=False)

source: 50 rows
root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)

output files: ['part-00000-tid-2693839663464382926-80d11f66-cfae-4b6a-b198-886fec9eb5ae-132-1-c000.csv']
read back: 50 rows
columns: ['customer_id', 'first_name', 'last_name', 'city', 'state', 'signup_date']
+-----------+----------+---------+---------+-----+-----------+
|customer_id|first_name|last_name|city     |state|signup_date|
+-----------+----------+---------+---------+-----+-----------+
|1          |Aarav     |Nair     |Bengaluru|KA   |2024-09-07 |
|2          |Vikram    |Reddy    |Mumbai   |MH   |2025-07-12 |
|3          |Varun     |Rao      |Bengaluru|KA   |2024-02-02 |
|4          |Rohan     |Patel    |Hyderabad|TG   |2024-08-26 |
|5          |Neha      |Sharma   |Mysuru   |KA   |2025-07-28 |
+-----------+----------+-

In [0]:
# Task 8: Orders by status and active customers

orders_by_status = (orders.groupBy("order_status")
                    .agg(F.count("order_id").alias("n_orders"))
                    .orderBy(F.desc("n_orders")))
orders_by_status.show()

# SQL on the temp view
spark.sql("""
    SELECT order_status, COUNT(*) AS n_orders
    FROM orders
    GROUP BY order_status
    ORDER BY n_orders DESC
""").show()

n_active = orders.select(F.countDistinct("customer_id").alias("n_customers")).collect()[0]["n_customers"]
print("distinct customers who placed at least one order:", n_active)

n_active_alt = orders.select("customer_id").distinct().count()
assert n_active == n_active_alt, "mismatch between countDistinct and distinct().count()!"
print("cross-check with distinct().count() passed:", n_active_alt)

+---------------+--------+
|   order_status|n_orders|
+---------------+--------+
|       COMPLETE|     141|
|         CLOSED|      51|
|        PENDING|      26|
|PENDING_PAYMENT|      25|
|       CANCELED|      16|
|SUSPECTED_FRAUD|      16|
|     PROCESSING|      13|
|        ON_HOLD|      12|
+---------------+--------+

+---------------+--------+
|   order_status|n_orders|
+---------------+--------+
|       COMPLETE|     141|
|         CLOSED|      51|
|        PENDING|      26|
|PENDING_PAYMENT|      25|
|       CANCELED|      16|
|SUSPECTED_FRAUD|      16|
|     PROCESSING|      13|
|        ON_HOLD|      12|
+---------------+--------+

distinct customers who placed at least one order: 51
cross-check with distinct().count() passed: 51


In [0]:
# Task 9: Top 5 customers by number of CLOSED orders

# DataFrame API:
top5_closed = (orders
               .filter(F.col("order_status") == "CLOSED")
               .groupBy("customer_id")
               .agg(F.count("order_id").alias("n_closed_orders"))
               .orderBy(F.desc("n_closed_orders"), F.desc("customer_id"))
               .limit(5))
top5_closed.show()

# SQL on the orders temp view
top5_closed_sql = spark.sql("""
    SELECT customer_id, COUNT(order_id) AS n_closed_orders
    FROM orders
    WHERE order_status = 'CLOSED'
    GROUP BY customer_id
    ORDER BY n_closed_orders DESC, customer_id DESC
    LIMIT 5
""")
top5_closed_sql.show()

# both approaches must agree
assert top5_closed.collect() == top5_closed_sql.collect(), "DataFrame and SQL results differ!"
print("DataFrame API and spark.sql() results match.")

+-----------+---------------+
|customer_id|n_closed_orders|
+-----------+---------------+
|          5|              4|
|         46|              3|
|         32|              3|
|         25|              3|
|         16|              3|
+-----------+---------------+

+-----------+---------------+
|customer_id|n_closed_orders|
+-----------+---------------+
|          5|              4|
|         46|              3|
|         32|              3|
|         25|              3|
|         16|              3|
+-----------+---------------+

DataFrame API and spark.sql() results match.


In [0]:
# Task 10: Filtering and sorting with nulls

# (a) PENDING* orders whose amount is null or below 2000
pending_low = (orders
               .filter((F.col("order_status").startswith("PENDING"))
                       & (F.col("amount").isNull() | (F.col("amount") < 2000)))
               .select("order_id", "order_status", "amount")
               .orderBy("order_id"))
pending_low.show()
print("count:", pending_low.count())

# cross-check via SQL on the orders temp view
spark.sql("""
    SELECT order_id, order_status, amount
    FROM orders
    WHERE order_status == 'PENDING%'
      AND (amount IS NULL OR amount < 2000)
    ORDER BY order_id
""").show()

# 10 most expensive orders
print("10 MOST EXPENSIVE:")
orders.orderBy(F.col("amount").desc()).select("order_id", "order_status", "amount").show(10)

# 10 cheapest orders, null values at the end
print("10 CHEAPEST (nulls last):")
cheapest = (orders.orderBy(F.col("amount").asc_nulls_last())
            .select("order_id", "order_status", "amount"))
cheapest.show(10)

# first 10 sorted rows must contain no null amounts
first10 = cheapest.limit(10).collect()
assert all(r["amount"] is not None for r in first10), "null leaked into cheapest 10!"
n_null = orders.filter(F.col("amount").isNull()).count()
print(f"null amounts in data: {n_null} -> all pushed to the very end by asc_nulls_last()")

+--------+---------------+-------+
|order_id|   order_status| amount|
+--------+---------------+-------+
|      15|PENDING_PAYMENT| 1338.6|
|     202|PENDING_PAYMENT|1147.92|
|     228|PENDING_PAYMENT|  892.4|
+--------+---------------+-------+

count: 3
+--------+------------+------+
|order_id|order_status|amount|
+--------+------------+------+
+--------+------------+------+

10 MOST EXPENSIVE:
+--------+---------------+--------+
|order_id|   order_status|  amount|
+--------+---------------+--------+
|     140|         CLOSED|49582.23|
|     132|        PENDING| 39998.5|
|      80|        PENDING| 36894.8|
|     164|SUSPECTED_FRAUD|36008.47|
|      90|       COMPLETE| 34371.4|
|     150|       COMPLETE|34360.25|
|     271|         CLOSED|33409.65|
|     106|SUSPECTED_FRAUD|32716.45|
|     247|       COMPLETE|32373.12|
|      47|       COMPLETE|30696.65|
+--------+---------------+--------+
only showing top 10 rows
10 CHEAPEST (nulls last):
+--------+---------------+-------+
|order_id| 

In [0]:
overall = orders.agg(
    F.count("order_id").alias("n_orders"),
    F.count("amount").alias("orders_with_amount"),   # count() skips nulls already
    F.round(F.sum("amount"), 2).alias("total_amount"),
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"))

overall.show()                                   

+--------+------------------+------------+----------+----------+----------+
|n_orders|orders_with_amount|total_amount|avg_amount|min_amount|max_amount|
+--------+------------------+------------+----------+----------+----------+
|     300|               290|  4022081.44|  13869.25|     446.2|  49582.23|
+--------+------------------+------------+----------+----------+----------+



In [0]:
# Task 12: Daily revenue with HAVING

daily_revenue = (orders
                 .groupBy("order_date")
                 .agg(F.count("order_id").alias("n_orders"),
                       F.round(F.sum("amount"), 2).alias("total_revenue"),
                       F.round(F.avg("amount"), 2).alias("avg_amount"))
                 .filter(F.col("total_revenue") > 100000)
                 .orderBy(F.desc("total_revenue")))
daily_revenue.show()

# cross-check with real HAVING on the orders temp view
spark.sql("""
    SELECT order_date,
           COUNT(order_id) AS n_orders,
           ROUND(SUM(amount), 2) AS total_revenue,
           ROUND(AVG(amount), 2) AS avg_amount
    FROM orders
    GROUP BY order_date
    HAVING SUM(amount) > 100000
    ORDER BY total_revenue DESC
""").show()

print("days above 100k revenue:", daily_revenue.count())

+----------+--------+-------------+----------+
|order_date|n_orders|total_revenue|avg_amount|
+----------+--------+-------------+----------+
|2026-07-27|       8|    143761.19|  17970.15|
|2026-07-26|      11|     138291.4|  13829.14|
|2026-08-24|       9|    130705.93|  14522.88|
|2026-07-29|       6|    127772.67|  21295.45|
|2026-07-09|       9|    126313.09|  14034.79|
|2026-08-07|       8|    122984.96|  20497.49|
|2026-08-05|       6|    113451.45|  18908.58|
|2026-08-22|       8|    110104.38|  13763.05|
+----------+--------+-------------+----------+

+----------+--------+-------------+----------+
|order_date|n_orders|total_revenue|avg_amount|
+----------+--------+-------------+----------+
|2026-07-27|       8|    143761.19|  17970.15|
|2026-07-26|      11|     138291.4|  13829.14|
|2026-08-24|       9|    130705.93|  14522.88|
|2026-07-29|       6|    127772.67|  21295.45|
|2026-07-09|       9|    126313.09|  14034.79|
|2026-08-07|       8|    122984.96|  20497.49|
|2026-08-05|